# Inital balance --> VWAP reasearch
## 1. data preperation
The initial reaseach data was imported from Ninjatrader's historical tick dataset. The research data was manually selected from the NQ daily chart using a daily adx <25, signifying a weak trend strength and a more consolidating market period

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
file_path = r"C:\Users\charl\Downloads\NQ 12-25.Last 28.08.25-06.10.25.txt"

df = pd.read_csv(
    file_path,
    sep=";",
    names=["date/time", "last", "bid", "ask", "volume"],
    skiprows=1,
    dtype={
        "date/time": "string",
        "last": "float32",
        "bid": "float32",
        "ask": "float32",
        "volume": "int32"
    }
)
df["date/time"] = pd.to_datetime(
    df["date/time"],
    format="%Y%m%d %H%M%S %f"
)
df = df.sort_values("date/time").reset_index(drop=True)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7062502 entries, 0 to 7062501
Data columns (total 5 columns):
 #   Column     Dtype         
---  ------     -----         
 0   date/time  datetime64[ns]
 1   last       float32       
 2   bid        float32       
 3   ask        float32       
 4   volume     int32         
dtypes: datetime64[ns](1), float32(3), int32(1)
memory usage: 161.6 MB


In [2]:
#converting from london timezone into New York using pythons standard IANA timezone database so trading session is aligned with U.S market hours
df["date/time"] = (
    df["date/time"]
    .dt.tz_localize("Europe/London")
    .dt.tz_convert("America/New_York")
)
print(df["date/time"].head())
print(df["date/time"].tail())

0   2025-09-13 21:02:32.364487900-04:00
1   2025-09-13 21:20:30.060631600-04:00
2   2025-09-13 21:20:30.637539800-04:00
3   2025-09-13 23:23:00.024702200-04:00
4   2025-09-14 14:00:00.462133700-04:00
Name: date/time, dtype: datetime64[ns, America/New_York]
7062497   2025-10-06 17:59:59.272000-04:00
7062498   2025-10-06 17:59:59.660000-04:00
7062499   2025-10-06 17:59:59.912000-04:00
7062500   2025-10-06 17:59:59.912000-04:00
7062501   2025-10-06 17:59:59.912000-04:00
Name: date/time, dtype: datetime64[ns, America/New_York]


In [3]:
#Using a seperate dataframe to only look at the New York cash session
rth = df[
    (df["date/time"].dt.time >= pd.Timestamp("09:30").time()) &
    (df["date/time"].dt.time < pd.Timestamp("16:00").time())
].copy()


## 2. Initial balance 
The initial balance (IB) is calculated as the maximum and minimum price during the first hour of the New York session (9:30-10:30am EST). For the backtest a 'live' updating IB is used. 
To check this was working correctly the IB was first calculated from a hindsight calculation based on the completed 9:30-10:30am period


In [4]:
#calcualting the initial balance(ib) for each day to be able to check with the live updating ib calculation that will be used in backtest
ib = rth[ #giving ib its own dataframe 
    (rth["date/time"].dt.time >= pd.Timestamp("09:30").time()) &
    (rth["date/time"].dt.time < pd.Timestamp("10:30").time())
].copy()
ib_daily = (
    ib.groupby(ib["date/time"].dt.date)
      .agg(
          ib_high_daily=("last", "max"),
          ib_low_daily=("last", "min")
      )
)

ib_daily["ib_range_daily"] = ib_daily["ib_high_daily"] - ib_daily["ib_low_daily"]
ib_daily = ib_daily.reset_index()

print((ib_daily).head(5))

    date/time  ib_high_daily  ib_low_daily  ib_range_daily
0  2025-09-15       24513.75      24478.25           35.50
1  2025-09-16       24556.75      24511.00           45.75
2  2025-09-17       24459.75      24404.00           55.75
3  2025-09-18       24816.00      24731.75           84.25
4  2025-09-19       24789.75      24719.50           70.25


In [5]:
#live IB calculation
ib_live = (
    (rth["date/time"].dt.time >= pd.Timestamp("09:30").time()) &
    (rth["date/time"].dt.time < pd.Timestamp("10:30").time())
)
rth["ib_high"] = (
    rth["last"]
    .where(ib_live)
    .groupby(rth["date/time"].dt.date)
    .cummax()
)

rth["ib_low"] = (
    rth["last"]
    .where(ib_live)
    .groupby(rth["date/time"].dt.date)
    .cummin()
)
#taking the last value < 10:30 as the final option for calculating ib
final_ib = ( 
    rth[ib_live]
    .groupby(rth.loc[ib_live, "date/time"].dt.date)[["ib_high", "ib_low"]]
    .last()
)

final_ib["ib_range"] = final_ib["ib_high"] - final_ib["ib_low"]
print(final_ib.head()) #checking live updating values match with previously found

             ib_high    ib_low  ib_range
date/time                               
2025-09-15  24513.75  24478.25     35.50
2025-09-16  24556.75  24511.00     45.75
2025-09-17  24459.75  24404.00     55.75
2025-09-18  24816.00  24731.75     84.25
2025-09-19  24789.75  24719.50     70.25


In [6]:
#making the ib only avaliable to use after 10:30 for the current rth session
rth["ib_set"] = (
    rth["date/time"].dt.time >= pd.Timestamp("10:30").time()
)
#adding the set ib values back into the rth session
rth["date"] = rth["date/time"].dt.date

rth = rth.merge(
    final_ib,
    left_on="date",
    right_index=True,
    how="left",
    suffixes=("", "_final")
)
print(rth)

                                  date/time      last       bid       ask  \
4       2025-09-14 14:00:00.462133700-04:00  24352.75  24346.25  24370.00   
80373      2025-09-15 09:30:00.024000-04:00  24497.50  24497.00  24497.50   
80374      2025-09-15 09:30:00.024000-04:00  24497.50  24497.00  24497.50   
80375      2025-09-15 09:30:00.024000-04:00  24497.25  24497.25  24497.50   
80376      2025-09-15 09:30:00.024000-04:00  24497.25  24497.25  24497.75   
...                                     ...       ...       ...       ...   
7058113    2025-10-06 15:59:59.472000-04:00  25182.75  25182.75  25184.00   
7058114    2025-10-06 15:59:59.624000-04:00  25183.50  25183.50  25184.00   
7058115    2025-10-06 15:59:59.696000-04:00  25184.00  25182.50  25184.00   
7058116    2025-10-06 15:59:59.696000-04:00  25184.00  25182.50  25184.00   
7058117    2025-10-06 15:59:59.696000-04:00  25184.00  25182.50  25184.00   

         volume  ib_high    ib_low  ib_set        date  ib_high_final  \
4 

## 3. VWAP
The Volume weighted average price (VWAP) is calcuated as the cumulative sum of price multiplied by volume, divided by the cumulative volume (VWAP=∑(Price×Volume)/∑Volume). This begins at the start of the New York cash session and runs 'live' through the session, updating as new price is recorded

In [7]:
rth["price_volume"] = rth["last"] * rth["volume"]

rth["cum_pv"] = ( 
    rth["price_volume"]
    .groupby(rth["date/time"].dt.date)
    .cumsum()
)

rth["cum_volume"] = (
    rth["volume"]
    .groupby(rth["date/time"].dt.date)
    .cumsum()
)
#making the VWAP rounded to the nearest NQ tick (0.25)
rth["vwap"] =(
    (rth["cum_pv"] / rth["cum_volume"])/0.25).round() *0.25
print(rth[["date/time", "last", "volume", "cum_pv", "cum_volume", "vwap"]].head(5))

                                date/time      last  volume    cum_pv  \
4     2025-09-14 14:00:00.462133700-04:00  24352.75       1  24352.75   
80373    2025-09-15 09:30:00.024000-04:00  24497.50       1  24497.50   
80374    2025-09-15 09:30:00.024000-04:00  24497.50       1  48995.00   
80375    2025-09-15 09:30:00.024000-04:00  24497.25       1  73492.25   
80376    2025-09-15 09:30:00.024000-04:00  24497.25       1  97989.50   

       cum_volume      vwap  
4               1  24352.75  
80373           1  24497.50  
80374           2  24497.50  
80375           3  24497.50  
80376           4  24497.50  


## 4. Building the IB → VWAP Mean Reversion Setup

For a trade setup to be valid, price must first break beyond the Initial Balance (IB) high or low and then return through the corresponding IB level. 

For an IB High setup, price must trade above the IB high before crossing back below it. For an IB Low setup, price must trade below the IB low before crossing back above it. 

The crossing back through the IB level is used as the entry event for the mean reversion trade towards VWAP. During the research phase every occassion this occurs is recorded, for part of the later backtest a more conservative apporach is tested where only one setup can happen at a time.


In [8]:
# IB high calculations
rth["above_ib_high"] = (
    rth["last"] > rth["ib_high_final"]
)

rth["had_ib_high_break"] = (
    rth["above_ib_high"]
    .groupby(rth["date/time"].dt.date)
    .cummax()
)

rth["ib_high_event"] = (
    rth["ib_set"] &
    rth["had_ib_high_break"] &
    (rth["last"].shift(1) > rth["ib_high_final"]) &
    (rth["last"] <= rth["ib_high_final"])
)
# IB low calculations
rth["below_ib_low"] = (
    rth["last"] < rth["ib_low_final"]
)

rth["had_ib_low_break"] = (
    rth["below_ib_low"]
    .groupby(rth["date/time"].dt.date)
    .cummax()
)

rth["ib_low_event"] = (
    rth["ib_set"] &
    rth["had_ib_low_break"] &
    (rth["last"].shift(1) < rth["ib_low_final"]) &
    (rth["last"] >= rth["ib_low_final"])
)

In [9]:
# Taking note of the number of events for IB high/low
high_events = rth[rth["ib_high_event"]].copy()
low_events = rth[rth["ib_low_event"]].copy()

print("High events:", len(high_events))
print("Low events:", len(low_events))
high_events["date"] = high_events["date/time"].dt.date
low_events["date"] = low_events["date/time"].dt.date

High events: 1233
Low events: 1254


## 5. Ib high --> VWAP/MAE
For each IB high event the maximum adverse excursion (MAE) is measured from the IB entry level until the VWAP is reached. This measures the maximum move against the short mean reversion trade before the VWAP is hit. If VWAP is not reached during the session, the MAE is measured until the end of the New York cash session and recored as that value.

The number of successful trades is calculated along with average and percentile MAE moves, these results will be used to determine a range of appropriate stop-loss level. The selceted range of stop-loss values will be tested on all the found setups to evaluate the trade off between allowing enough room for adverse movements and limiting losses.

In [10]:
high_results = []

for date, day in rth.groupby(rth["date/time"].dt.date):

    events = day[day["ib_high_event"]]

    for idx in events.index:

        event_position = day.index.get_loc(idx)
        future = day.iloc[event_position + 1:]

        if future.empty:
            continue

        entry = day.loc[idx, "ib_high_final"]

        vwap_hit = (
            future["last"].to_numpy()
            <= future["vwap"].to_numpy()
        )

        if vwap_hit.any():

            hit_position = vwap_hit.argmax()
            hit = future.iloc[hit_position]

            # Only measure movement against the trade before the VWAP is hit
            
            path = future.iloc[:hit_position + 1]

            mae = path["last"].max() - entry

            high_results.append({
                "date": date,
                "event_time": day.loc[idx, "date/time"],
                "entry": entry,
                "vwap": hit["vwap"],
                "vwap_hit": True,
                "exit_time": hit["date/time"],
                "mae_to_vwap": mae
            })

        else:  #VWAP was not hit during remainder of the New York cash session

            mae = future["last"].max() - entry

            high_results.append({
                "date": date,
                "event_time": day.loc[idx, "date/time"],
                "entry": entry,
                "vwap": future["vwap"].iloc[-1],
                "vwap_hit": False,
                "exit_time": future["date/time"].iloc[-1],
                "mae_to_vwap": mae
            })

high_results = pd.DataFrame(high_results)

# Remove duplicate events that happen at exactly the same time and price 
high_results = (
    high_results
    .drop_duplicates(
        subset=["date", "event_time", "entry"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("High events:", len(high_results))
print("VWAP hit rate:", high_results["vwap_hit"].mean())

High events: 1128
VWAP hit rate: 0.5540780141843972


In [11]:
successful_highs = high_results[
    high_results["vwap_hit"] == True
].copy()

successful_highs["mae_to_vwap"] = 0.0

for date, day in rth.groupby(rth["date/time"].dt.date):

    events = successful_highs[
        successful_highs["date"] == date
    ]

    for idx in events.index:

        event_time = successful_highs.loc[idx, "event_time"]
        entry = successful_highs.loc[idx, "entry"]

        future = day[
            day["date/time"] > event_time
        ]

        vwap_hit = (
            future["last"].to_numpy()
            <= future["vwap"].to_numpy()
        )

        hit_position = vwap_hit.argmax()

        path = future.iloc[:hit_position + 1]

        mae = path["last"].max() - entry

        successful_highs.loc[idx, "mae_to_vwap"] = mae
print("Successful trades:", len(successful_highs))
print("Average move against trade:", successful_highs["mae_to_vwap"].mean())
print("Median move against trade:", successful_highs["mae_to_vwap"].median())
print("75th percentile:", successful_highs["mae_to_vwap"].quantile(0.75))
print("90th percentile:", successful_highs["mae_to_vwap"].quantile(0.90))
print("Maximum move against trade:", successful_highs["mae_to_vwap"].max())

Successful trades: 625
Average move against trade: 15.528
Median move against trade: 6.25
75th percentile: 20.25
90th percentile: 40.0
Maximum move against trade: 76.25


In [12]:
stop_sizes = [10, 20, 25, 30, 40, 50, 60, 100]

stop_results = []

for stop in stop_sizes:

    winner_mask = (
        (high_results["vwap_hit"] == True) &
        (high_results["mae_to_vwap"] <= stop)
    )

    winners = high_results[winner_mask]

    wins = len(winners)
    losses = len(high_results) - wins

    # Profit on successful VWAP trades
    winning_pnl = winners["entry"] - winners["vwap"]

    # Loss on trades stopped out
    losing_pnl = -stop * losses

    total_pnl = winning_pnl.sum() + losing_pnl

    expectancy = total_pnl / len(high_results)

    stop_results.append({
        "stop": stop,
        "wins": wins,
        "losses": losses,
        "win_rate": wins / len(high_results),
        "avg_win": winning_pnl.mean(),
        "avg_loss": stop,
        "expectancy": expectancy,
        "total_pnl": total_pnl
    })

stop_results = pd.DataFrame(stop_results)

print(stop_results)

   stop  wins  losses  win_rate    avg_win  avg_loss  expectancy  total_pnl
0    10   367     761  0.325355  10.814714        10   -3.227837   -3641.00
1    20   435     693  0.385638  11.224138        20   -7.958777   -8977.50
2    25   531     597  0.470745  11.040960        25   -8.033910   -9062.25
3    30   531     597  0.470745  11.040960        30  -10.680186  -12047.25
4    40   564     564  0.500000  11.287234        40  -14.356383  -16194.00
5    50   564     564  0.500000  11.287234        50  -19.356383  -21834.00
6    60   564     564  0.500000  11.287234        60  -24.356383  -27474.00
7   100   625     503  0.554078  11.288800       100  -38.337323  -43244.50


##  6. Ib high findings 
For the initial research data, the IB High → VWAP results are fairly poor. Across all of the stop-loss values tested, the setup produces a loss. The results also show that giving the trade more room with a larger stop-loss does not improve performance and instead leads to larger losses. 

These results indicate that the IB high --> vwap setup does not perform well for this data set so wont be used in the backtest. However, the information on stop-loss values will be kept in mind for the rest of the research


## 7. IB low --> VWAP/MAE
The IB low --> VWAP setup uses the same methodology as the IB high --> VWAP setup, but in the opposite direction. For each event, the MAE is measured from the entry level until the VWAP is hit. If VWAP is not reached, the MAE is measured until the end of the New York cash session and that value is recorded. The Ib low --> VWAP is a long mean reversion, so the MAE is measured as the maximum move below the IB low 

The same methods are used to measure and record the MAE, which is used to determine the range of stop-loss values. This then allows the trade off between allowing room for adverse movements and limitng losses to be evaluated for the IB low --> VWAP setup

In [13]:
low_results = []

for date, day in rth.groupby(rth["date/time"].dt.date):

    events = day[day["ib_low_event"]]

    for idx in events.index:

        event_position = day.index.get_loc(idx)
        future = day.iloc[event_position + 1:]

        if future.empty:
            continue

        entry = day.loc[idx, "ib_low_final"]

        vwap_hit = (
            future["last"].to_numpy()
            >= future["vwap"].to_numpy()
        )

        if vwap_hit.any():

            hit_position = vwap_hit.argmax()
            hit = future.iloc[hit_position]
            
            # Only measure movement against the trade before the VWAP is hit
            path = future.iloc[:hit_position + 1]
            
            mae = entry - path["last"].min()

            low_results.append({
                "date": date,
                "event_time": day.loc[idx, "date/time"],
                "entry": entry,
                "vwap": hit["vwap"],
                "vwap_hit": True,
                "exit_time": hit["date/time"],
                "mae_to_vwap": mae
            })

        else:     #VWAP was not hit during remainder of the New York cash session


            mae = entry - future["last"].min()

            low_results.append({
                "date": date,
                "event_time": day.loc[idx, "date/time"],
                "entry": entry,
                "vwap": future["vwap"].iloc[-1],
                "vwap_hit": False,
                "exit_time": future["date/time"].iloc[-1],
                "mae_to_vwap": mae
            })

low_results = pd.DataFrame(low_results)

# Remove duplicate events caused by multiple ticks
low_results = (
    low_results
    .drop_duplicates(
        subset=["date", "event_time", "entry"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("low events:", len(low_results))
print("VWAP hit rate:", low_results["vwap_hit"].mean())


low events: 1150
VWAP hit rate: 0.9391304347826087


In [14]:
successful_lows = low_results[
    low_results["vwap_hit"] == True
].copy()

successful_lows["mae_to_vwap"] = 0.0

for date, day in rth.groupby(rth["date/time"].dt.date):

    events = successful_lows[
        successful_lows["date"] == date
    ]

    for idx in events.index:

        event_time = successful_lows.loc[idx, "event_time"]
        entry = successful_lows.loc[idx, "entry"]

        future = day[
            day["date/time"] > event_time
        ]

        vwap_hit = (
            future["last"].to_numpy()
            >= future["vwap"].to_numpy()
        )

        hit_position = vwap_hit.argmax()

        path = future.iloc[:hit_position + 1]

        mae = entry - path["last"].min()

        successful_lows.loc[idx, "mae_to_vwap"] = mae


print("Successful trades:", len(successful_lows))
print("Average move against trade:", successful_lows["mae_to_vwap"].mean())
print("Median move against trade:", successful_lows["mae_to_vwap"].median())
print("75th percentile:", successful_lows["mae_to_vwap"].quantile(0.75))
print("90th percentile:", successful_lows["mae_to_vwap"].quantile(0.90))
print("Maximum move against trade:", successful_lows["mae_to_vwap"].max())

Successful trades: 1080
Average move against trade: 27.690046296296295
Median move against trade: 11.5
75th percentile: 17.25
90th percentile: 73.5
Maximum move against trade: 162.0


In [15]:
stop_sizes = [10, 20, 25, 30, 40, 50, 60, 100]

stop_results = []

for stop in stop_sizes:

    winner_mask = (
        (low_results["vwap_hit"] == True) &
        (low_results["mae_to_vwap"] <= stop)
    )

    winners = low_results[winner_mask]

    wins = len(winners)
    losses = len(low_results) - wins

    # Profit on successful VWAP trades
    winning_pnl = winners["vwap"] - winners["entry"]

    # Loss on trades stopped out
    losing_pnl = -stop * losses

    total_pnl = winning_pnl.sum() + losing_pnl

    expectancy = total_pnl / len(low_results)

    stop_results.append({
        "stop": stop,
        "wins": wins,
        "losses": losses,
        "win_rate": wins / len(low_results),
        "avg_win": winning_pnl.mean(),
        "avg_loss": stop,
        "expectancy": expectancy,
        "total_pnl": total_pnl
    })

stop_results = pd.DataFrame(stop_results)

print(stop_results)

   stop  wins  losses  win_rate    avg_win  avg_loss  expectancy  total_pnl
0    10   434     716  0.377391  21.714862        10    1.968913    2264.25
1    20   820     330  0.713043  21.329573        20    9.469783   10890.25
2    25   844     306  0.733913  21.995557        25    9.490652   10914.25
3    30   848     302  0.737391  21.940153        30    8.300217    9545.25
4    40   848     302  0.737391  21.940153        40    5.674130    6525.25
5    50   848     302  0.737391  21.940153        50    3.048043    3505.25
6    60   848     302  0.737391  21.940153        60    0.421957     485.25
7   100   977     173  0.849565  19.620522       100    1.625435    1869.25


## 8. IB low findings 
For the intial research data, the IB low --> VWAP results were promising. Across all stop loss values the return was positive. The stop loss values of 20 and 25 mesured the highest expected gain and total pnl.

Given the much more positive results in the IB low setup compared to IB high setup, the IB low --> VWAP setup will be used in the backtest. A 20-point stop-loss will be used as it provides a suitable trade-off between allowing room for adverse movement and limiting losses, based on the MAE results for both IB low and IB high setups.